# 04. Read a Deep-Learning Paper Like an Engineer

**Author:** Md. Mobarak Karim, Ph.D.  
**Level:** Beginner → research-practical  
**Course:** Deep Learning for Optical Imaging

This is a core research notebook. It teaches how to convert a paper into an implementation specification and a reuse decision.

> **How to study this notebook:** read the explanation first, predict what the code should do, run it, change one parameter, and explain why the result changed.


## Learning objectives

- Read a paper in three passes
- Extract a complete implementation specification
- Translate figures/equations into code plans
- Audit splits and metrics
- Maintain a discrepancy log
- Choose reuse/adapt/reimplement/reject


## Mind map

```mermaid
mindmap
  root((Paper to implementation))
    Scientific claim
      Input
      Target
      Population
      Baseline
    Data pipeline
      Split unit
      Preprocessing
      Registration
      Augmentation
    Model
      Architecture
      Shapes
      Activation
      Loss
    Training
      Optimizer
      LR
      Scheduler
      Checkpoint
    Evaluation
      Metric definition
      Aggregation
      External validation
    Decision
      Reuse
      Adapt
      Reimplement
      Reject

```


## 1. The purpose of reading a paper for implementation

A normal literature read asks: *What did the authors conclude?*

An engineering read asks additional questions:

- What exactly went into the model?
- What exactly came out?
- Which biological unit defined the split?
- What preprocessing happened before the network saw the image?
- Which parts are equations versus implementation choices?
- Which details are in the supplement/code but not the main text?
- What would I need to reproduce one reported figure or metric?

The output of your reading should be a **paper specification**, not a vague summary.


## 2. Three-pass paper-reading workflow

### Pass 1 — Scientific claim
Write one sentence for each:

| Field | Question |
|---|---|
| Problem | What scientific/engineering problem is solved? |
| Input | What modality, dimensions, channels, sampling? |
| Target | Mask, class, high-quality image, H&E, fluorescence, parameter map? |
| Population | Which specimens/patients/animals? |
| Claim | What is better than what? |
| Baseline | What are the comparison methods? |
| Validation | Internal only or external/generalization? |

If you cannot fill this table, do not start coding.

### Pass 2 — Implementation specification
Extract architecture, preprocessing, losses, optimizer, schedule, split, metrics, patch size, augmentation, checkpoint rule, and hardware.

### Pass 3 — Discrepancy search
Compare main paper, supplement, README, configs, code, released weights, and metric scripts. Record contradictions.


## 3. Build a paper specification sheet

Use this template for every serious paper:

```text
Paper:
Repository:
License:
Task:
Input modality:
Input shape:
Target:
Independent biological unit:
Train/val/test split:
Preprocessing:
Normalization:
Patch extraction:
Augmentation:
Architecture:
Output activation:
Loss equation:
Loss weights:
Optimizer:
Learning rate:
Scheduler:
Batch size:
Epochs:
Checkpoint rule:
Metrics:
Postprocessing:
Pretrained weights:
Hardware:
Software versions:
Reported main result:
External validation:
Known limitations:
```

A filled specification is your bridge from prose to code.


## 4. Read figures as implementation information

Architecture figures often hide important decisions:

- number of encoder/decoder levels;
- channel widths;
- kernel sizes;
- stride/downsampling;
- skip connections;
- normalization layers;
- activation functions;
- residual blocks;
- attention modules;
- output head;
- input/output resolution.

### Example: translating a figure into a table

| Stage | Input shape | Operation | Channels | Output shape |
|---|---|---|---:|---|
| input | 1×256×256 | — | 1 | 1×256×256 |
| enc1 | 1×256×256 | 2×Conv3×3 + ReLU | 32 | 32×256×256 |
| pool1 | 32×256×256 | MaxPool2 | 32 | 32×128×128 |
| enc2 | 32×128×128 | 2×Conv3×3 + ReLU | 64 | 64×128×128 |

If the paper figure cannot be converted into an unambiguous table, note the uncertainty rather than inventing details.


## 5. Translate every equation into a coding plan

Suppose the paper defines:

```text
L_total = lambda1 * L1 + lambda2 * L_struct
```

Do not immediately write one combined expression. First document:

```text
L1:
  input: prediction, target
  expected range: non-negative
  reduction: mean?
L_struct:
  exact equation?
  window size?
  data range?
lambda1:
  value?
lambda2:
  value?
```

Then implement and test each component separately.


In [ ]:
import torch
import torch.nn.functional as F

prediction = torch.tensor([0.1, 0.7, 0.4])
target = torch.tensor([0.0, 1.0, 0.5])

# Term 1: test independently
loss_l1 = F.l1_loss(prediction, target)

# Term 2: placeholder structural term for teaching.
# In a real reproduction, implement the exact equation from the paper.
loss_struct = torch.mean((prediction - target) ** 2)

lambda_l1 = 1.0
lambda_struct = 0.2

loss_total = lambda_l1 * loss_l1 + lambda_struct * loss_struct

print("L1:", loss_l1.item())
print("struct:", loss_struct.item())
print("total:", loss_total.item())


## 6. Read the Methods for hidden data operations

Look specifically for:

- clipping;
- log transform;
- percentile normalization;
- resizing/interpolation;
- registration;
- background subtraction;
- optical correction;
- stain normalization;
- patch rejection;
- blank-tile filtering;
- artifact removal;
- target generation;
- manual annotation rules.

These steps can have a larger effect than architecture changes.

### Optical-imaging example
A virtual-staining paper may say "registered paired images." You need to know:
- rigid, affine, or deformable registration?
- at what resolution?
- were poorly aligned pairs excluded?
- was registration done before or after patching?
- did test images use the same registration procedure?


## 7. Read the split as carefully as the model

Ask:

```text
What was randomized?
patient?
eye?
animal?
slide?
tissue block?
image?
patch?
frame?
```

A phrase like "random 80/10/10 split" is incomplete unless the randomized unit is stated.

### Example
If one patient contributes 500 pathology tiles, a random tile split can put 400 tiles in training and 50 highly related tiles in test. That is not a strong estimate of patient-level generalization.


## 8. Metrics: copy the definition, not just the name

"Dice", "SSIM", "PSNR", "accuracy", or "AUROC" can be implemented differently.

Record:
- per-image or global aggregation?
- foreground class only?
- threshold used?
- empty masks handled how?
- data range for PSNR/SSIM?
- macro or micro averaging?
- confidence intervals?
- specimen-level or patch-level averaging?

If your metric implementation differs, your number may not be comparable even when the model is identical.


## 9. Read ablation studies as causal clues

An ablation asks what happens when one component is removed or changed. Use ablations to decide which parts are essential to reproduce.

Example:

| Variant | Dice |
|---|---:|
| baseline U-Net | 0.82 |
| + attention | 0.83 |
| + special loss | 0.87 |
| + attention + special loss | 0.88 |

This suggests the special loss may matter more than the attention block. Start reproduction effort accordingly.


## 10. Worked fictional example: OCT layer segmentation paper

Imagine a paper claims improved retinal-layer segmentation.

### Scientific specification

```text
Input: OCT B-scan, 1×512×1024
Target: 8-class layer mask
Independent unit: eye
Preprocessing: crop retina, percentile scale to [0,1]
Model: residual U-Net
Loss: weighted CE + Dice
Training: AdamW, lr=3e-4, 150 epochs
Selection: best validation mean Dice
Test: unseen eyes
Metric: per-class Dice averaged across eyes
```

### Questions before reuse

1. Do your OCT images have similar axial/lateral sampling?
2. Same scanner/vendor?
3. Same pathology distribution?
4. Same number/class definition of layers?
5. Same crop/flattening?
6. Are pretrained weights licensed?
7. Does code implement the same weighted CE described in the paper?
8. Does the test script average per B-scan or per eye?

Only after these questions do you know whether "use the code" is a sensible plan.


## 11. Discrepancy log

Create a table whenever paper and code disagree:

| Item | Paper says | Code says | Decision | Evidence |
|---|---|---|---|---|
| LR | 1e-4 | 3e-4 | test both / ask authors | config file |
| resize | 256 | 512 | use code for reproduction | dataloader |
| Dice smoothing | not stated | 1e-5 | document | metrics.py |

Never silently resolve discrepancies. They become part of your reproducibility record.


## 12. Decision after reading

At the end of the paper read, choose one:

- **REUSE** — code closely matches your task and is reproducible.
- **ADAPT** — parts are useful, but preprocessing/model/output must change.
- **REIMPLEMENT** — concept is useful but released code is unsuitable/incomplete.
- **REJECT** — method does not match your scientific problem or cannot be validated adequately.

The point of reading is not to admire the architecture. The point is to make a defensible implementation decision.


## Practice exercise + worked answer

### Exercise
A fluorescence-denoising paper reports 35 dB PSNR. Public code exists, but:
- training data are synthetic Gaussian-noise images;
- your data have Poisson-dominated shot noise and camera read noise;
- the repository has no license;
- pretrained weights are available;
- the architecture is clearly described.

What should you do?

### Worked reasoning
Do **not** directly use the weights in your research pipeline. The noise model does not match your acquisition, and the missing license creates reuse uncertainty. A better plan is:

1. use the paper as a methodological reference;
2. verify the architecture/loss from the paper;
3. reimplement a clean baseline;
4. simulate or acquire noise that matches your fluorescence system;
5. validate on paired or controlled real data;
6. compare against classical denoising and no-denoising baselines;
7. inspect whether dim structures are removed.


## End-of-notebook checklist

Before moving on, you should be able to explain the main ideas **without looking at the code**. If you cannot explain why a method, loss, split, or metric is appropriate, repeat the relevant section before using it in research.
